# LFM2 — Basic Usage

## Imports

In [1]:
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [2]:
MODEL_ID = "LiquidAI/LFM2-700M"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Architecture: ['Lfm2ForCausalLM']
Parameters: 742,489,344
Device: mps:0
Dtype: torch.bfloat16


## Single Turn Generation

In [3]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [4]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
pprint(inputs, sort_dicts=False, width=120)

{'input_ids': tensor([[    1,     6,  6423,   708,  3493,   856,   779,  5706,   803,  4481,
           540,     7,   708,     6, 64015,   708]], device='mps:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}


In [5]:
outputs = model.generate(**inputs, max_new_tokens=128)
print(outputs)

tensor([[    1,     6,  6423,   708,  3493,   856,   779,  5706,   803,  4481,
           540,     7,   708,     6, 64015,   708,  1098,  5706,   803,  4481,
           856,  5242,   523,  1311,   856,   768,  2938,  3771,   810,  4545,
          6055,   875,  1822,   521, 12688,   521, 11411,  1711,  5797,   521,
          5009,   521,   810,  3886,   523,  5242,   856, 35248,   875, 60297,
          1559,   906,   779,   908,  3890,   808, 25810,   521,  5765, 19508,
          7242,   521, 27344, 50013, 33334,   521,   810,   779, 13824, 10169,
         33252,   523,  1311,   856,  1236,  2538,   875,  1352,  6506,   884,
         13032,   521, 12688,   521,   810,   779, 13083, 12169,   523,     7]],
       device='mps:0')


In [6]:
print(tokenizer.decode(outputs[0]))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|startoftext|><|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris. It is a major city and global center for art, fashion, gastronomy, culture, and education. Paris is renowned for landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and the historic Latin Quarter. It is also known for its influence on politics, fashion, and the arts worldwide.<|im_end|>


In [7]:
input_len = inputs["input_ids"].shape[-1]
response = tokenizer.decode(outputs[0][input_len:])
print(response)

The capital of France is Paris. It is a major city and global center for art, fashion, gastronomy, culture, and education. Paris is renowned for landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and the historic Latin Quarter. It is also known for its influence on politics, fashion, and the arts worldwide.<|im_end|>


In [8]:
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)

The capital of France is Paris. It is a major city and global center for art, fashion, gastronomy, culture, and education. Paris is renowned for landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and the historic Latin Quarter. It is also known for its influence on politics, fashion, and the arts worldwide.


## System Prompt

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>system
You are a helpful assistant who responds in all capitals.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [10]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]
outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

PARIS IS THE CAPITAL OF FRANCE.


## Multi-Turn Generation

In [11]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The capital of France is Paris. It's a major global city and a leading cultural, economic, and political center in Europe. Paris is renowned for its historical landmarks, museums, fashion, gastronomy, and romantic ambiance.


In [12]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant',
  'content': "The capital of France is Paris. It's a major global city and a leading cultural, economic, and political "
             'center in Europe. Paris is renowned for its historical landmarks, museums, fashion, gastronomy, and '
             'romantic ambiance.'}]


In [13]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant',
  'content': "The capital of France is Paris. It's a major global city and a leading cultural, economic, and political "
             'center in Europe. Paris is renowned for its historical landmarks, museums, fashion, gastronomy, and '
             'romantic ambiance.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [14]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

One of the most famous landmarks in Paris is the Eiffel Tower. Built for the 1889 Exposition Universelle (World's Fair), it stands at 324 meters tall and offers breathtaking views of the city from its observation decks. It's an iconic symbol of France and one of the most recognizable structures in the world.


## Streaming Generation

In [15]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)

streamer = transformers.TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
outputs = model.generate(**inputs, max_new_tokens=128, streamer=streamer)

The capital of France is Paris. It is a major city and global center for art, fashion, gastronomy, culture, and education. Paris is renowned for landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and the historic Latin Quarter. It is also known for its influence on politics, fashion, and the arts worldwide.
